Fase 1: Exploración y Diagnóstico (Data Profiling)

In [0]:
# Carga una tabla desde un archivo delimitado.
df = spark.table('workspace.electiva.caso_practico_ventas').toPandas()

In [0]:
# Mostrar las primeras 10 filas
display(df.head(10))

# Mostrar las últimas 5 filas
display(df.tail(5))

# Mostrar el número de registros y columnas
print("Número de registros:", df.shape[0])
print("Número de columnas:", df.shape[1])

In [0]:
# Mostrar información general del DataFrame
df.info()

# Mostrar los tipos de datos de cada columna
print(df.dtypes)

In [0]:
# Genera métricas estadísticas generales (media, min, max, etc.)
df.describe()

In [0]:
import pandas as pd

# Calcular el número de valores nulos por columna
nulos = df.isnull().sum()

# Calcular el porcentaje de valores nulos
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100

# Crear una tabla de diagnóstico
diagnostico_nulos = pd.DataFrame({
    "Valores nulos": nulos,
    "Porcentaje (%)": porcentaje_nulos.round(2)
})

display(diagnostico_nulos)
# se identifican valores faltantes en tres variables. Estos datos deberán tratarse en la fase de limpieza, ya que pueden afectar los cálculos estadísticos y los análisis posteriores.

FASE 2. Limpieza y depuración (Data Cleaning)

In [0]:
# Identificar filas completamente duplicadas
print("Filas completamente duplicadas:",
      df.duplicated().sum())

# Identificar identificadores de transacción repetidos
print("Transacciones con ID repetido:",
      df["id_transaccion"].duplicated().sum())

# Eliminar registros con identificadores repetidos
df = df.drop_duplicates(
    subset=["id_transaccion"],
    keep="first"
).copy()

# Verificar el resultado
print("Registros después de eliminar duplicados:", len(df))
print("IDs duplicados restantes:",
      df["id_transaccion"].duplicated().sum())
      #la eliminación evita contabilizar dos veces una misma transacción. Se conserva la primera aparición de cada identificador, suponiendo que los registros repetidos corresponden a duplicaciones y no a ventas diferentes.

In [0]:
# Revisar los métodos de pago existentes
print(df["metodo_pago"].value_counts())

# Unificar la categoría inconsistente
df["metodo_pago"] = df["metodo_pago"].replace({
    "Tarjeta Credito": "Tarjeta de Crédito"
})

# Comprobar los cambios
print(df["metodo_pago"].value_counts())

In [0]:
# Calcular la mediana de cantidad
mediana_cantidad = df["cantidad"].median()

# Reemplazar los nulos de cantidad por la mediana
df["cantidad"] = df["cantidad"].fillna(mediana_cantidad)

# Calcular la moda de satisfacción
moda_satisfaccion = df["satisfaccion_cliente"].mode()[0]

# Reemplazar los nulos de satisfacción por la moda
df["satisfaccion_cliente"] = df[
    "satisfaccion_cliente"
].fillna(moda_satisfaccion)

# Mostrar los valores nulos restantes
print(df[["cantidad", "satisfaccion_cliente"]].isnull().sum())

In [0]:
# Localizar la transacción con precio atípico
outlier = df[df["precio_unitario"] == 9999.00]

display(outlier)

# Obtener la categoría del producto afectado
categoria_outlier = outlier["categoria_producto"].iloc[0]

# Calcular el precio promedio de esa categoría,
# excluyendo el valor atípico
precio_medio_categoria = df.loc[
    (df["categoria_producto"] == categoria_outlier) &
    (df["precio_unitario"] != 9999.00),
    "precio_unitario"
].mean()

# Reemplazar el valor atípico
df.loc[
    df["precio_unitario"] == 9999.00,
    "precio_unitario"
] = precio_medio_categoria

# Comprobar el resultado
print("Categoría afectada:", categoria_outlier)
print("Precio imputado:", round(precio_medio_categoria, 2))
print("Valores de 9999 restantes:",
      (df["precio_unitario"] == 9999.00).sum())
      # esta decisión permite conservar la transacción en el conjunto de datos y reduce la influencia del precio extremo sobre las estadísticas y los ingresos calculados posteriormente. Se asume que el precio de 9.999 USD es un error o un valor anómalo; antes de una aplicación empresarial real, convendría verificarlo con la fuente original.

In [0]:
# Verificar la estructura del conjunto de datos limpio
print("Dimensiones finales:", df.shape)

# Comprobar valores faltantes
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Comprobar duplicados
print("\nDuplicados restantes:",
      df["id_transaccion"].duplicated().sum())

# Mostrar una muestra de los datos limpios
display(df.head(10))
# un conjunto de datos con 120 registros y 9 columnas, sin valores faltantes en las variables tratadas, sin IDs repetidos y con los métodos de pago normalizados y el precio atípico sustituido.

FASE 3. Transformación e ingeniería de atributos

In [0]:
# Calcular el monto total de cada transacción
df["monto_total"] = (
    df["cantidad"] * df["precio_unitario"]
)

# Mostrar las columnas y los resultados
display(df[[
    "id_transaccion",
    "cantidad",
    "precio_unitario",
    "monto_total"
]].head(10))
# Se crea una nueva columna llamada monto_total, que representa el valor total de cada venta. Para calcularla, se multiplica la cantidad de unidades vendidas por el precio unitario del producto. La operación es vectorial, es decir, se ejecuta automáticamente para todas las filas del DataFrame, sin necesidad de utilizar un ciclo.

In [0]:
# Convertir la columna fecha a formato datetime
df["fecha"] = pd.to_datetime(
    df["fecha"],
    errors="coerce"
)

# Crear una columna con el día de la semana
df["dia_semana"] = df["fecha"].dt.day_name()

# Crear una columna con el mes
df["mes"] = df["fecha"].dt.month

# Visualizar los resultados
display(df[[
    "fecha",
    "dia_semana",
    "mes"
]].head(10))
# pd.to_datetime() convierte la columna de texto en un formato de fecha y tiempo.
# errors="coerce" convierte fechas no válidas en valores NaT (fecha faltante).
# .dt.day_name() extrae el nombre del día de la semana.
# .dt.month extrae el número del mes.
# la conversión permite realizar análisis temporales, como identificar los días con mayores ventas o comparar los ingresos por mes. En caso de que alguna fecha resulte inválida, se deberá revisar antes de utilizarla en el análisis.

In [0]:
import numpy as np

# Definir las condiciones de clasificación
condiciones = [
    df["monto_total"] < 100,
    (df["monto_total"] >= 100) &
    (df["monto_total"] <= 500),
    df["monto_total"] > 500
]

# Definir las categorías
categorias = ["Bajo", "Medio", "Alto"]

# Crear la nueva columna de clasificación
df["rango_monto"] = np.select(
    condiciones,
    categorias,
    default="Sin clasificar"
)

# Mostrar los resultados
display(df[[
    "id_transaccion",
    "monto_total",
    "rango_monto"
]].head(15))

# Contar las transacciones por rango
print(df["rango_monto"].value_counts())
# a clasificación facilita el análisis de la distribución de las ventas, permitiendo conocer cuántas transacciones pertenecen a cada rango de ingresos. También permite comparar posteriormente los resultados por región, categoría o método de pago.
# Resultado esperado: una nueva columna llamada rango_monto, que identifica las transacciones como bajas, medias o altas.

FASE 4. Agregación y resumen de datos

In [0]:
# Agrupar por categoría y calcular ingresos y unidades
resumen_categoria = df.groupby(
    "categoria_producto"
).agg(
    ingresos_totales=("monto_total", "sum"),
    promedio_unidades=("cantidad", "mean")
).reset_index()

# Redondear los resultados
resumen_categoria["ingresos_totales"] = (
    resumen_categoria["ingresos_totales"].round(2)
)

resumen_categoria["promedio_unidades"] = (
    resumen_categoria["promedio_unidades"].round(2)
)

# Mostrar la tabla resumen
display(resumen_categoria)
# este análisis permite identificar cómo se distribuyen los ingresos entre las categorías de productos y comparar sus volúmenes promedio de venta. Una categoría puede tener ingresos elevados debido a precios unitarios altos, aunque su promedio de unidades vendidas no sea el mayor.


explicacion:
groupby("categoria_producto") agrupa las transacciones por categoría.

agg() permite calcular varias estadísticas al mismo tiempo.

sum calcula los ingresos totales de cada categoría.

mean calcula el promedio de unidades vendidas por transacción.

reset_index() convierte el resultado en una tabla con índice convencional.

In [0]:
# Crear tabla dinámica de ingresos por región y categoría
tabla_dinamica = pd.pivot_table(
    df,
    values="monto_total",
    index="region",
    columns="categoria_producto",
    aggfunc="sum",
    fill_value=0
)

# Mostrar la tabla dinámica
display(tabla_dinamica)
# permite comparar los ingresos generados por cada categoría en las distintas regiones. Por ejemplo, se puede observar qué categorías concentran mayores ingresos en cada zona geográfica y detectar diferencias en la distribución comercial.

explicacion:
pd.pivot_table() crea una tabla cruzada a partir de los datos.

values="monto_total" indica la variable que se va a resumir.

index="region" establece las regiones en las filas.

columns="categoria_producto" coloca las categorías en las columnas.

aggfunc="sum" suma los montos de las transacciones.

fill_value=0 reemplaza las combinaciones sin registros por cero.

In [0]:
# Calcular la satisfacción promedio por método de pago
satisfaccion_pago = df.groupby(
    "metodo_pago"
).agg(
    satisfaccion_promedio=(
        "satisfaccion_cliente",
        "mean"
    ),
    total_transacciones=(
        "id_transaccion",
        "count"
    )
).reset_index()

# Redondear el promedio
satisfaccion_pago["satisfaccion_promedio"] = (
    satisfaccion_pago["satisfaccion_promedio"].round(2)
)

# Mostrar los resultados
display(satisfaccion_pago)
# Se agrupan las transacciones según el método de pago y se calcula el promedio de satisfacción de los clientes en cada grupo. También se cuenta el número de transacciones para conocer el tamaño de cada grupo.
# permite comparar los niveles medios de satisfacción asociados a los diferentes métodos de pago. Sin embargo, estos resultados muestran asociaciones descriptivas y no demuestran que el método de pago sea la causa de una mayor o menor satisfacción.

FASE 5. Selección, filtrado y muestreo

In [0]:
# Filtrar ventas de la región Norte
# con satisfacción igual o superior a 4
ventas_norte_satisfechas = df[
    (df["region"] == "Norte") &
    (df["satisfaccion_cliente"] >= 4)
].copy()

# Mostrar los registros filtrados
display(ventas_norte_satisfechas)

# Mostrar el número de registros seleccionados
print(
    "Total de ventas filtradas:",
    len(ventas_norte_satisfechas)
)
# este filtro permite estudiar las ventas de una región específica con niveles altos de satisfacción, facilitando el análisis de sus características comerciales.

df["region"] == "Norte" selecciona las ventas realizadas en la región Norte.

df["satisfaccion_cliente"] >= 4 selecciona las transacciones con calificaciones de satisfacción de 4 o 5.

El operador & exige que ambas condiciones se cumplan.

In [0]:
# Extraer una muestra aleatoria del 10% por región
muestra_estratificada = df.groupby(
    "region",
    group_keys=False
).sample(
    frac=0.10,
    random_state=42
)

# Mostrar la muestra
display(muestra_estratificada)

# Mostrar el tamaño de la muestra
print("Total de registros en la muestra:",
      len(muestra_estratificada))

# Comparar la distribución por región
comparacion_regiones = pd.DataFrame({
    "Datos originales": df["region"].value_counts(),
    "Muestra": muestra_estratificada["region"].value_counts()
}).fillna(0).astype(int)

display(comparacion_regiones)
# el muestreo estratificado permite trabajar con un subconjunto de datos que mantiene la presencia de las diferentes regiones. Esto puede resultar útil cuando se requiere realizar análisis preliminares con menos registros, procurando evitar que alguna región quede sin representación.

groupby("region") divide los datos en grupos según la región.

sample(frac=0.10) selecciona aleatoriamente el 10 % de los registros de cada grupo.

random_state=42 permite reproducir la misma selección aleatoria cuando se ejecuta el código con los mismos datos.

La tabla de comparación permite verificar cuántos registros hay en cada región antes y después del muestreo.